# CIFAR-10 Dataset

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

torch.manual_seed(42)

In [31]:
# Defining Transforms
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert PIL Image to tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize
])

In [32]:
# Downloading and Loading CIFAR10 dataset
trainset = CIFAR10(
    root='./data',  # Directory to store data
    train=True,     # Load training set
    download=False,  # Download if not already available
    transform=transform,
)
trainloader = DataLoader(
    trainset,
    batch_size=100,  # Adjust batch size as needed
    shuffle=True    # Shuffle the data
)

In [33]:
testset = CIFAR10(
    root='./data',
    train=False,    # Load test set
    download=False,
    transform=transform
)
testloader = DataLoader(
    testset,
    batch_size=100,
    shuffle=False   # No need to shuffle test data
)

In [34]:
# Checking the size of the dataset
len(trainset), len(testset)

(50000, 10000)

In [35]:
# Defining the model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, (5, 5)) # 3 input channels, 6 output channels, 5x5 kernel
        self.pool = nn.MaxPool2d((2, 2), 2) # 2x2 kernel, stride 2
        self.conv2 = nn.Conv2d(6, 16, (5, 5)) # 6 input channels, 16 output channels, 5x5 kernel
        self.fc1 = nn.Linear(16 * 5 * 5, 120) # 16 channels, 5x5 image size
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        
        # Calculate and print total number of parameters
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Total number of parameters: {total_params}")
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) # Conv1 -> ReLU -> Pool
        x = self.pool(F.relu(self.conv2(x))) # Conv2 -> ReLU -> Pool
        x = x.view(-1, 16 * 5 * 5) # Flatten
        # Pass through fully connected layers
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [36]:
# Initialize the model and optimizer
model = Net()
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

Total number of parameters: 62006


In [37]:
# Running a forward pass for 1 batch
for images, labels in trainloader:
    optimizer.zero_grad()
    output = model(images)
    loss_val = loss(output, labels)
    break
print(f"Inital Loss: {loss_val.item()}")

Inital Loss: 2.3038909435272217


In [38]:
# Training the model for 2 epochs
for epoch in range(2):
    running_loss = 0
    for data in trainloader:
        inputs, labels = data
        optimizer.zero_grad() # Zero the gradients
        outputs = model(inputs) # Get predictions
        loss_val = loss(outputs, labels) # Calculate loss
        loss_val.backward() # Backpropagate
        optimizer.step() # Update weights
        
        running_loss += loss_val.item() # Accumulate loss
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")

Epoch 1, Loss: 1.915567151069641
Epoch 2, Loss: 1.4591114513874055


In [40]:
# Accuracy on test set
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct / total}%")

Accuracy: 50.66%
